# Mi mapa de pistas de Vengadores: Doomsday

**Categoría: EDA. Autor: Roberto Cantero. Corte: 24/11/2026.**

Quiero saber qué afirmaciones he encontrado en una muestra de cobertura del tráiler y dónde termina el detalle descrito y empieza una interpretación. No predigo la película. No he completado una revisión plano a plano.

## Mi ruta de lectura

Primero reviso el origen de las fichas, después cuento tipos y cobertura, y termino comprobando dependencia y límites.

- Una ficha es una afirmación seleccionada, no un secreto confirmado.
- Detalle reportado: una página describe algo del avance.
- Hipótesis: una explicación que necesita pruebas adicionales.
- Contexto oficial: lo anunciado por Disney.
- Solapamiento: fichas que mencionan dos páginas. No demuestra independencia.

## De dónde salen mis datos

He seleccionado seis páginas porque cubren el tráiler final. No he hecho una búsqueda exhaustiva ni aleatoria. Disney+ y Good Morning America pertenecen al mismo grupo; cinco grupos editoriales tampoco equivalen a cinco observaciones independientes. Todos pueden interpretar el mismo vídeo.

Cada fila conserva título, afirmación, límite, fuentes, localizadores, categoría, región temática y estado de revisión. `segundo` queda vacío porque no he verificado inicios de plano. `segundo_transcripcion` solo ayuda a localizar cuatro diálogos mediante subtítulos automáticos. Esos subtítulos contienen errores: no los uso para verificar nombres propios ni identidades.

In [1]:
from pathlib import Path
import json, collections, hashlib
BASE = Path.cwd()
if not (BASE / 'pistas.json').exists():
    BASE = Path(__file__).resolve().parent if '__file__' in globals() else BASE
pistas = json.loads((BASE/'pistas.json').read_text(encoding='utf-8'))
fuentes = json.loads((BASE/'fuentes.json').read_text(encoding='utf-8'))
# compruebo las unidades y las referencias antes de contar
ids = {f['id'] for f in fuentes}
assert len({p['id'] for p in pistas}) == len(pistas)
assert all(set(p['fuentes'].split(';')) <= ids for p in pistas)
assert all(p['estado'] in {'detalle_reportado','hipotesis','contexto_oficial'} for p in pistas)
assert all(p['segundo'] is None for p in pistas)
assert all(set(p['depende_de']) <= {q['id'] for q in pistas} for p in pistas)
print(f'Cargo {len(pistas)} afirmaciones y {len(fuentes)} páginas. No tengo un inventario de planos.')


Cargo 24 afirmaciones y 6 páginas. No tengo un inventario de planos.


## Qué voy a contar

Cuento fichas, no personas, planos ni verdades. Una ficha puede citar varias páginas: por eso sus menciones no suman 24. Comparo categorías sin convertir su porcentaje en una probabilidad.

In [2]:
# cuento afirmaciones y compruebo cuánto depende la selección de una sola página
estados = dict(collections.Counter(p['estado'] for p in pistas))
por_fuente = {f['nombre']:sum(f['id'] in p['fuentes'].split(';') for p in pistas) for f in fuentes}
una_fuente = [p['id'] for p in pistas if len(p['fuentes'].split(';')) == 1]
hipotesis = [p for p in pistas if p['estado']=='hipotesis']
grupos = {f['id']:f['grupo_editorial'] for f in fuentes}
grupos_por_pista = {p['id']:len({grupos[f] for f in p['fuentes'].split(';')}) for p in pistas}
solapamientos = []
for i,a in enumerate(fuentes):
    for b in fuentes[i+1:]:
        sa={p['id'] for p in pistas if a['id'] in p['fuentes'].split(';')}
        sb={p['id'] for p in pistas if b['id'] in p['fuentes'].split(';')}
        solapamientos.append({'a':a['nombre'],'b':b['nombre'],'comunes':len(sa & sb),'union':len(sa | sb)})
resultado = {'fecha_corte':'2026-11-24','categoria':'EDA','unidad':'afirmación seleccionada',
    'total_fichas':len(pistas),'estados':estados,'fichas_por_fuente':por_fuente,
    'fichas_con_una_sola_pagina':una_fuente,'hipotesis_con_una_sola_pagina':sum(len(p['fuentes'].split(';'))==1 for p in hipotesis),
    'grupos_editoriales':len(set(grupos.values())),'grupos_por_pista':grupos_por_pista,
    'solapamientos':solapamientos,'revision_visual_completa':0,
    'pistas_con_tiempo_de_subtitulo':sum(p['segundo_transcripcion'] is not None for p in pistas),
    'sha256_pistas':hashlib.sha256((BASE/'pistas.json').read_bytes()).hexdigest()}
(BASE/'resultados.json').write_text(json.dumps(resultado,ensure_ascii=False,indent=2),encoding='utf-8')
print('Recuento por tipo:', estados)
print('Fichas con una sola página:',len(una_fuente), 'de',len(pistas))
print('Hipótesis con una sola página:',resultado['hipotesis_con_una_sola_pagina'],'de',len(hipotesis))
print('Páginas por grupo editorial:',len(fuentes),'páginas /',resultado['grupos_editoriales'],'grupos')
for f in fuentes:
    quedan=[p for p in pistas if set(p['fuentes'].split(';'))-{f['id']}]
    print('Al retirar',f['nombre'],'conservo',len(quedan),'fichas con al menos otra página.')


Recuento por tipo: {'detalle_reportado': 16, 'hipotesis': 6, 'contexto_oficial': 2}
Fichas con una sola página: 9 de 24
Hipótesis con una sola página: 4 de 6
Páginas por grupo editorial: 6 páginas / 5 grupos
Al retirar Disney+ conservo 22 fichas con al menos otra página.
Al retirar Good Morning America conservo 23 fichas con al menos otra página.
Al retirar GamesRadar+ conservo 22 fichas con al menos otra página.
Al retirar SlashFilm conservo 22 fichas con al menos otra página.
Al retirar ScreenCrush conservo 23 fichas con al menos otra página.
Al retirar Ars Technica conservo 23 fichas con al menos otra página.


In [3]:
# muestro una tabla legible sin instalar paquetes externos
print('TIPO                    FICHAS    PORCENTAJE')
for tipo,n in resultado['estados'].items():
    print(f'{tipo:24} {n:3}       {100*n/len(pistas):5.1f}%')
print('\nPÁGINA / FICHAS EN LAS QUE LA CITO')
for fuente,n in resultado['fichas_por_fuente'].items():print(f'{fuente}: {n}')


TIPO                    FICHAS    PORCENTAJE
detalle_reportado         16        66.7%
hipotesis                  6        25.0%
contexto_oficial           2         8.3%

PÁGINA / FICHAS EN LAS QUE LA CITO
Disney+: 2
Good Morning America: 7
GamesRadar+: 14
SlashFilm: 9
ScreenCrush: 3
Ars Technica: 8


## Mi lectura

Identifico 16 detalles reportados, seis hipótesis y dos contextos oficiales. Que una cuarta parte del inventario sean hipótesis describe mi selección; no significa que una cuarta parte del tráiler sea especulativa. No hay fichas con revisión visual completa. Los tiempos de subtítulos ayudan a volver al diálogo, pero no sustituyen esa revisión.

## Cuánto cambia al retirar una página

La prueba anterior elimina cada página de las referencias y cuenta qué fichas perderían todo respaldo documental en esta muestra. Mide dependencia de mi selección; no mide credibilidad del medio. Dos páginas del mismo grupo pueden repetir una afirmación y dos grupos diferentes también pueden copiarse.

## Límites y explicaciones alternativas

He seleccionado temas llamativos, así que el inventario puede sobrerrepresentar teorías. No he visionado los vídeos de los analistas. Los rótulos de las páginas y mi clasificación pueden diferir: cuando un medio presenta una causalidad como segura, la mantengo como hipótesis si falta confirmación primaria. La fecha de corte, los localizadores y las decisiones permiten revisar esa elección.

## Mi conclusión

Puedo mostrar de dónde procede cada afirmación y cuáles dependen de una lectura adicional. No puedo asegurar que este sea un inventario completo de secretos ni resolver la trama. Para ampliar el análisis registraré cada plano con inicio, final, observación directa y pruebas de identidad; después separaré esa nueva tabla de la cobertura publicada.